# 05 · Análisis final — tablas, intervalos de confianza y figuras
Junta todas las corridas (B-01/B-02 y el barrido del 04) y produce lo que va al informe y al póster:

1. Tabla final de AUC por video (media ± desviación entre semillas) por γ y manipulación, con la caída relativa frente a in-domain.
2. **Δ AUC (γ − BCE) con IC 95 % por bootstrap emparejado de videos.** Así se separa un efecto real de la variación entre semillas y entre videos de test.
3. Curvas del % de ejemplos fáciles (p_t > 0.9) por época y γ: el diagnóstico del mecanismo.

**No necesita GPU ni datos de FF++**: solo los JSON y las predicciones (IDs y probabilidades). Busca corridas en `/kaggle/input` (output del 03 y del 04) y en `results/metrics/` del repo. También corre en local, desde `notebooks/kaggle/`.

Sin predicciones (por ejemplo, B-01/B-02 transcritos de la v5), la corrida entra en la tabla pero no en el bootstrap.

In [ ]:
# Kaggle: clona el repo (Internet = On). En local: usa la copia del repo desde notebooks/kaggle/
import os, sys, subprocess
from pathlib import Path
REPO_URL = "https://github.com/Gredith/DeepLearning-G8-TA_Repository.git"
ROOT = Path("/kaggle/working/repo") if Path("/kaggle").exists() else Path.cwd().parents[1]
if Path("/kaggle").exists() and not ROOT.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src" / "kaggle_e2"))
print("ROOT =", ROOT)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import analysis as A, plotstyle as S; S.apply()
fuentes = [ROOT / "results" / "metrics"] + [Path(d) for d in os.environ.get("RUNS_DIRS", "").split(os.pathsep) if d]
if Path("/kaggle/input").exists():
    fuentes += sorted({p.parent for p in Path("/kaggle/input").rglob("*.json") if A.parse_run_name(p.stem)[0] is not None})
runs = A.load_runs(fuentes)
OUT = Path("/kaggle/working/resultados") if Path("/kaggle").exists() else ROOT / "results" / "metrics" / "final"
OUT.mkdir(parents=True, exist_ok=True)
print(f"{len(runs)} corridas;", sum(r["_preds"] is not None for r in runs.values()), "con predicciones")
pd.DataFrame([{"run": n, "gamma": r["gamma"], "seed": r["seed"], "preds": r["_preds"] is not None} for n, r in runs.items()]).sort_values(["gamma", "seed"])

## 1. Tabla final

In [ ]:
tabla = A.metrics_table(runs); tabla.to_csv(OUT / "tabla_metricas_por_corrida.csv", index=False)
res = A.summary_by_gamma(tabla, "auc_video"); res.to_csv(OUT / "tabla_final_auc_video.csv", index=False)
caida = A.summary_by_gamma(tabla, "caida_rel_video_pct"); caida.to_csv(OUT / "tabla_final_caida_rel_video.csv", index=False)
print("AUC por video (media ± desv. entre semillas)")
display(res.pivot(index="test", columns="gamma", values="texto"))
print("Caída relativa frente a in-domain (%)")
caida.pivot(index="test", columns="gamma", values="media").round(1)

In [ ]:
tests = [t for t in A.TEST_ORDER if t in set(res.test)]
gammas = sorted(res.gamma.unique())
fig, ax = plt.subplots(figsize=(8.5, 4))
w = 0.8 / max(len(gammas), 1); x = np.arange(len(tests))
for i, (g, col) in enumerate(zip(gammas, S.SERIES)):
    sub = res[res.gamma == g].set_index("test").reindex(tests)
    ax.bar(x + (i - (len(gammas) - 1) / 2) * w, sub.media, w * 0.95, yerr=sub.desv.fillna(0), capsize=3, color=col,
           label=("BCE (γ=0)" if g == 0 else f"Focal γ={g:g}") + f" · n={int(sub.n_semillas.max())}")
ax.axhline(0.5, color=S.MUTED, lw=1, ls="--"); ax.text(len(tests) - 0.5, 0.51, "azar", color=S.MUTED, fontsize=8, ha="right")
ax.set_xticks(x, [t + ("\n(in-domain)" if t == "Deepfakes" else "") for t in tests]); ax.set_ylim(0, 1.05)
ax.set_ylabel("AUC por video"); ax.grid(axis="x", visible=False)
ax.set_title("Entrenado solo en Deepfakes c23: AUC por manipulación de test", loc="left")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=min(len(gammas), 4))
plt.tight_layout(); plt.savefig(OUT / "fig_final_auc_por_gamma.png", dpi=200); plt.show()

## 2. ¿La diferencia con BCE es real? Bootstrap emparejado (2000 réplicas)

In [ ]:
boot = []
for g in [g for g in gammas if g != 0]:
    for t in tests:
        r = A.paired_bootstrap_delta(runs, g, 0.0, test=t, n_boot=int(os.environ.get("N_BOOT", 2000)))
        if r: boot.append(r)
boot = pd.DataFrame(boot)
if len(boot):
    boot.to_csv(OUT / "bootstrap_delta_auc_video.csv", index=False)
    display(boot.round(4))
    fig, ax = plt.subplots(figsize=(8, 0.5 + 0.35 * len(boot)))
    lab = boot.apply(lambda r: f"γ={r.gamma_a:g} · {r.test} (n={r.semillas})", axis=1)
    ax.errorbar(boot.delta_auc_video, np.arange(len(boot)), xerr=[boot.delta_auc_video - boot.ic95_bajo, boot.ic95_alto - boot.delta_auc_video],
                fmt="o", color=S.SERIES[1], capsize=3)
    ax.axvline(0, color=S.MUTED, lw=1); ax.set_yticks(np.arange(len(boot)), lab); ax.invert_yaxis()
    ax.set_xlabel("Δ AUC por video frente a BCE (IC 95 %)"); ax.set_title("Efecto de Focal Loss sobre BCE", loc="left")
    plt.tight_layout(); plt.savefig(OUT / "fig_final_bootstrap_delta.png", dpi=200); plt.show()
else:
    print("No hay pares de corridas con predicciones (preds_*.csv.gz): correr el 03 o el 04 con la versión actual de train.py.")

## 3. Mecanismo: % de ejemplos fáciles y AUC de validación por época

In [ ]:
hist = A.history_table(runs)
if len(hist):
    hist.to_csv(OUT / "historial_por_epoca.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    for g, col in zip(gammas, S.SERIES):
        h = hist[hist.gamma == g].groupby("epoch")
        lab = "BCE (γ=0)" if g == 0 else f"Focal γ={g:g}"
        for ax, m in zip(axes, ["train_pct_faciles", "val_auc_frame"]):
            mu, sd = h[m].mean(), h[m].std(ddof=1).fillna(0)
            ax.plot(mu.index, mu, marker="o", ms=5, color=col, label=lab); ax.fill_between(mu.index, mu - sd, mu + sd, color=col, alpha=0.15)
    axes[0].set_title("% de ejemplos fáciles en train (p_t > 0.9)", loc="left"); axes[1].set_title("AUC de validación (Deepfakes)", loc="left")
    for a in axes: a.set_xlabel("época"); a.axvline(1.5, color=S.GRID, lw=1); a.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(OUT / "fig_final_mecanismo.png", dpi=200); plt.show()

## 4. Lectura (completar con los números de arriba)
| Pregunta | Evidencia | Conclusión |
|---|---|---|
| ¿Focal γ=2 mejora el cross-manipulation frente a BCE? | Δ AUC e IC 95 % de la sección 2 | |
| ¿La relación con γ es no monótona (H-B2)? ¿γ=5 baja el in-domain? | Tabla de la sección 1 | |
| ¿El mecanismo ocurre (menos fáciles con γ mayor)? | Sección 3 | |
| ¿FaceSwap < 0.5 se repite entre semillas y pérdidas? | Tabla de la sección 1 | Diagnóstico en el notebook 06 |

Regla: si el IC 95 % incluye 0, no se afirma un efecto, se reporta como nulo.